In [1]:
import pandas as pd
import sys
from pathlib import Path

# DEFINIR PATH Y REQUIREMENT

csv_path = 'database/cisco_oferta.csv'
requirement = pd.DataFrame([{
    'code': 'requirement',      # 100 ports at 800G
    '100': 16,
    '40': 16,
    '25': 64,
    '10': 64,
}])

In [2]:
sys.path.insert(0, str(Path('.').resolve()))

from model.modular import Modular
df = pd.read_csv(csv_path)
modules_df = df[df['type'] == 'modular']
linecards_df = df[df['type'] == 'linecard']
print(f"Found {len(modules_df)} module rows, {linecards_df['code'].nunique()} unique linecards")


Found 7 module rows, 27 unique linecards


In [3]:
module_codes = modules_df['code'].unique()
results = {}
errors = {}

for module_code in module_codes:
    try:
        module_data = df[df['code'] == module_code]
        module_family = module_data['family'].iloc[0]
        linecards_for_family = df[(df['type'] == 'linecard') & (df['family'] == module_family)]
        combined_data = pd.concat([module_data, linecards_for_family], ignore_index=True)
        modular = Modular(combined_data)
        solution = modular.apply_heuristic(requirement)
        if solution is not None and not solution.empty:
            results[module_code] = solution
            print(f"✓ {module_code}: Found solution with {len(solution)} linecards")
        else:
            errors[module_code] = "Requirement cannot be satisfied within maxmodules limit"
            print(f"✗ {module_code}: {errors[module_code]}")
    except Exception as e:
        errors[module_code] = str(e)
        print(f"✗ {module_code}: Error - {str(e)[:100]}")

print(f"\n{'='*60}")
print(f"Summary: {len(results)} successful, {len(errors)} failed")

✗ 9808: Error - the solution exceeds maxmodules
✗ 9804: Error - the solution exceeds maxmodules
✓ 9516: Found solution with 2 linecards
✓ 9508: Found solution with 2 linecards
✓ 9504: Found solution with 2 linecards
✗ 9400: Error - the solution exceeds maxmodules
✗ 9400 : Error - the solution exceeds maxmodules

Summary: 3 successful, 4 failed


In [4]:
if results:
    for module_code, solution in results.items():
        print(f"\nModule: {module_code}")
        print(f"\nLinecard configuration:")

        speed_columns = [col for col in solution.columns if col not in ['code', 'value']]

        speed_columns_sorted = sorted(speed_columns, key=lambda x: float(x) if x not in ['code', 'value'] else 0, reverse=True)
        display_cols = ['code'] + speed_columns_sorted + ['value']

        display_solution = solution[display_cols].copy()
        display_solution = display_solution.fillna(0)
        for col in speed_columns_sorted + ['value']:
            display_solution[col] = display_solution[col].astype(int)

        print(display_solution.to_string(index=False))

        total_value = int(solution['value'].sum())
        print(f"\nTotal value (throughput*ports): {total_value}")
        print("-" * 60)
else:
    print("No successful configurations found.")

if errors:
    print("\n\nFAILED MODULES:")
    for module_code, error in errors.items():
        print(f"  {module_code}: {error}")


Module: 9516

Linecard configuration:
          code  400  100  50  40  25  10  1  value
N9K-X9736C-FX3    0   16   0   0  64  16  0   3360
 N9K-X9716D-GX    0    0   0  16   0  48  0   1120

Total value (throughput*ports): 4480
------------------------------------------------------------

Module: 9508

Linecard configuration:
          code  400  100  50  40  25  10  1  value
N9K-X9736C-FX3    0   16   0   0  64  16  0   3360
 N9K-X9716D-GX    0    0   0  16   0  48  0   1120

Total value (throughput*ports): 4480
------------------------------------------------------------

Module: 9504

Linecard configuration:
          code  400  100  50  40  25  10  1  value
N9K-X9736C-FX3    0   16   0   0  64  16  0   3360
 N9K-X9716D-GX    0    0   0  16   0  48  0   1120

Total value (throughput*ports): 4480
------------------------------------------------------------


FAILED MODULES:
  9808: the solution exceeds maxmodules
  9804: the solution exceeds maxmodules
  9400: the solution exceeds 